# AI Sommelier RAG: 와인 리뷰 기반 와인 추천 서비스

In [1]:
%pip install -qU langchain-pinecone pinecone

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
TEXT_MODEL = 'gpt-4.1-mini'
VISION_MODEL = 'gpt-4.1-mini'
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM = 1536
PINECONE_INDEX_NAME = 'winemeg-review-data'
PINECONE_CLOUD = 'aws'
PINECONE_REGION = 'us-east-1'

## Pinecone Vector DB 준비

In [6]:
from pinecone import Pinecone, ServerlessSpec

# .env에 PINECONE_API_KEY 설정이 추가 되고 환경 변수가 로드 되어야 함
pc = Pinecone()

# 같은 이름의 Index가 없을 때만 새로 생성한다.
if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION
        )
    )
    print("Pinecone Index 생성 완료:", PINECONE_INDEX_NAME)
else:
    print("이미 존재하는 Pinecone Index 사용:", PINECONE_INDEX_NAME)

pinecone_index = pc.Index(PINECONE_INDEX_NAME)
pinecone_index.describe_index_stats()

이미 존재하는 Pinecone Index 사용: winemeg-review-data


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

## 와인 리뷰 CSV를 Document로 변환

In [7]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path='data/winemag-data-130k-v2.csv',
    encoding='utf-8'
)

all_docs = loader.load()

print("전체 Document 수 : ", len(all_docs))

전체 Document 수 :  65499


In [8]:
# 전체 데이터 저장시 시간, 비용이 많이 들기 때문에 제한해서 사용한다
MAX_DOCS = 3000

docs = all_docs if MAX_DOCS is None else all_docs[:MAX_DOCS]

print("이번 테스트에서 사용할 Document 수 : ", len(docs))

for i, doc in enumerate(docs[:2], start=1):
    print(f"[Document {i}]")
    print("meatdata : ", doc.metadata)
    print(doc.page_content)

이번 테스트에서 사용할 Document 수 :  3000
[Document 1]
meatdata :  {'source': 'data/winemag-data-130k-v2.csv', 'row': 0}
: 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia
[Document 2]
meatdata :  {'source': 'data/winemag-data-130k-v2.csv', 'row': 1}
: 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15
province: Douro
region_1: 
region_2: 
taster_name: Roger Vo

## Pinecone Vector Store 연결

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = PineconeVectorStore(
    index=pinecone_index,
    embedding=embeddings
)

print("PineconeVectorStore 연결 완료")

PineconeVectorStore 연결 완료


## Document 를 Pinecone에 저장

In [11]:
from tqdm.auto import tqdm 

BATCH_SIZE = 100

for start in tqdm(range(0, len(docs), BATCH_SIZE)):
    end = start + BATCH_SIZE
    batch = docs[start:end]

    # row 번호를 사용하면 같은 데이터를 다시 실행해도 같은 id가 만들어진다.
    ids = [
        f"winemag-{doc.metadata.get('row', start + offset)}"
        for offset, doc in enumerate(batch)
    ]

    vector_store.add_documents(
        documents=batch,
        ids=ids
    )

print("Pinecone 저장 완료")
print(pinecone_index.describe_index_stats())

  0%|          | 0/30 [00:00<?, ?it/s]

Pinecone 저장 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3000}},
 'total_vector_count': 3000,
 'vector_type': 'dense'}


## 검색 결과 검증
- 검색어를 직접 넣어보고 의도한 와인 리뷰가 검색 되는지 확인 (RAG 구현 전단계)

In [12]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k" : 3}
)

test_queries = [
    'full-bodied red wine with dark fruit and oak',
    'fresh white wine with citrus and high acidity',
    'sweet dessert wine with honey aroma'
]

for query in test_queries:
    print("검색어 : ", query)
    results = retriever.invoke(query)

    for i, doc in enumerate(results, start=1):
        print(f"[검색결과 {i}]")
        print("metadata : ", doc.metadata)
        print(doc.page_content[:1000])

    print("=" * 100)

검색어 :  full-bodied red wine with dark fruit and oak
[검색결과 1]
metadata :  {'row': 1354.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 1354
country: US
description: Strong in black currant, raisin, blackberry pie and oaky flavors, this full-bodied effort has thick tannins. It's pleasant for drinking now with barbecue and roasts.
designation: 
points: 86
price: 30
province: California
region_1: California
region_2: California Other
taster_name: 
taster_twitter_handle: 
title: Dark Hundred 2011 Red (California)
variety: Red Blend
winery: Dark Hundred
[검색결과 2]
metadata :  {'row': 1771.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 1771
country: US
description: A very deep red color catches the attention. Next, strong, wild aromas like smoke and black rubber lead to a rather fruity but still firm and tannic texture. The flavors are more like blackberries and blueberries, but the overall effect is slightly viscous.
designation: 
points: 84
price: 9
province: California
region_1: California


## LLM 단독 추천 확인

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=TEXT_MODEL
)

simple_recommend_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 친절한 소믈리에입니다. 음식과 잘 어울리는 와인을 초보자도 이해하기 쉽게 추천하세요."
    ),
    (
        "human",
        "다음 음식에 어울리는 와인을 추천해주세요.\n\n음식: {dish}"
    )
])

simple_recommend_chian = simple_recommend_prompt | llm | StrOutputParser()

print(simple_recommend_chian.invoke({"dish" : "로즈마리를 곁들인 스테이크"}))


로즈마리를 곁들인 스테이크에는 풍미가 깊고 탄닌이 적당한 레드 와인이 잘 어울려요. 초보자도 부담 없이 즐길 수 있는 와인으로는 다음을 추천드립니다:

- 까베르네 소비뇽 (Cabernet Sauvignon): 풍부한 과일 향과 부드러운 탄닌이 스테이크의 육즙과 잘 어울려요.
- 멀롯 (Merlot): 부드럽고 라운드한 맛이 로즈마리 향과 조화를 이뤄 부담 없이 즐길 수 있어요.
- 쉬라즈 (Shiraz): 스파이시한 향이 로즈마리의 허브 향과 잘 맞아 특별한 맛을 느낄 수 있습니다.

특히 멀롯은 부드럽고 무겁지 않아 와인 초보자에게 가장 추천드립니다. 맛있게 즐기세요!


## 음식 설명으로 와인 리뷰 검색

In [20]:
sample_dish_flavor = (
    "A juicy grilled steak with rosemary aroma, roasted vegetables, "
    "savory meat flavor, and a rich smoky finish"
)

sample_docs = retriever.invoke(sample_dish_flavor)

for i, doc in enumerate(sample_docs, start=1):
    print(f"[검색결과 {i}]")
    print("metadata : ", doc.metadata)
    print(doc.page_content[:1000])
    print("=" * 100)

[검색결과 1]
metadata :  {'row': 264.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 264
country: South Africa
description: A good amount of earthy spice, tea leaves and forrest floor lead the way on the nose, but the black cherry and berry fruit aromas follow shortly after with an additional accent of sweet cured meat. Medium weight and lush, the creamy mouth transitions into a finish loaded with sweet spice and bittersweet cocoa.
designation: 
points: 89
price: 19
province: Stellenbosch
region_1: 
region_2: 
taster_name: Lauren Buzzeo
taster_twitter_handle: @laurbuzz
title: Jardin 2007 Syrah (Stellenbosch)
variety: Syrah
winery: Jardin
[검색결과 2]
metadata :  {'row': 1374.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 1374
country: US
description: Made in a soft, gentle way, this pretty Mourvèdre has chocolate-infused blackberry, currant, raspberry, licorice, cola and pepper flavors. It's very dry, and will drink well with a char-broiled steak.
designation: 
points: 86
price: 24
province: 

In [18]:
def format_wine_docs(docs: list) -> str:
    """검색된 와인 리뷰 Document를 추천 Prompt에 넣기 좋은 문자열로 변환하는 함수"""

    formatted = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        row = doc.metadata.get("row", "unknown")

        formatted.append(
            f"[와인리뷰 {i}]\n"
            f"source : {source}\n"
            f"row : {row}\n"
            f"content :\n{doc.page_content}"
        )

    return "\n\n".join(formatted)    

## 음식 이미지를 풍미 설명으로 변환

In [21]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnableLambda

vision_llm = ChatOpenAI(
    model=VISION_MODEL
)

def describe_dish_flavor(query: dict) -> str:
    """음식 이미지 URL 목록을 받아 와인 페어링에 필요한 풍미 설명을 문자열로 반환한다."""

    image_urls = query.get("image_urls", [])

    if not image_urls:
        raise ValueError("image_urls 값이 비어 있습니다.")

    content = [
        {
            "type": "text",
            "text": (
                "Look at the food image(s) and describe the dish for wine pairing. "
                "Focus on ingredients, cooking method, sauce, texture, intensity, acidity, fat, sweetness, "
                "spiciness, and overall flavor profile. "
                "Write the answer in English because the wine review data is in English."
                "Keep it concise in 2-3 sentences and include searchable flavor keywords."
            ),
        }
    ]

    # 여러 장의 이미지를 받을 수 있도록 URL 목록을 반복 처리한다.
    for image_url in image_urls:
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": image_url},
            }
        )

    messages = [
        SystemMessage(
            content=(
                "You are a culinary expert who writes concise and useful flavor descriptions "
                "for wine pairing search queries."
            )
        ),
        HumanMessage(content=content),
    ]

    response = vision_llm.invoke(messages)
    return response.content


describe_dish_flavor_chain = RunnableLambda(describe_dish_flavor)

dish_flavor = describe_dish_flavor_chain.invoke({
    "image_urls": [
        "https://justcook.butcherbox.com/wp-content/uploads/2025/02/Rib-Eye-Steak-au-Poivre-with-Roasted-Veggies--500x500.jpg"
    ]
})

print(dish_flavor)

The dish features a pepper-crusted grilled steak cooked to medium-rare, offering a savory, juicy, and slightly charred flavor with moderate fat and robust intensity. It is paired with grilled zucchini, bell peppers, and onions, adding a smoky, slightly sweet, and earthy vegetable complement. Flavor keywords: grilled, pepper-crusted, medium-rare, smoky, savory, juicy, charred, earthy, slightly sweet.


## VISION MODEL이 반환한 설명으로 관련 와인 리뷰 검색

In [22]:
def search_wines(dish_flavor: str) -> dict:
    """음식 풍미 설명과 유사한 와인 리뷰를 Pinecone에서 검색한다."""

    docs = retriever.invoke(dish_flavor)

    return {
        "dish_flavor" : dish_flavor,
        "wine_reviews" : format_wine_docs(docs),
        "retrieved_docs" : docs
    }

wine_review_retrieval_chain = RunnableLambda(search_wines)
retrieval_result = wine_review_retrieval_chain.invoke(dish_flavor)

print("[음식설명]")
print(retrieval_result["dish_flavor"])

print("\n[검색 된 와인 리뷰]")
print(retrieval_result["wine_reviews"])

[음식설명]
The dish features a pepper-crusted grilled steak cooked to medium-rare, offering a savory, juicy, and slightly charred flavor with moderate fat and robust intensity. It is paired with grilled zucchini, bell peppers, and onions, adding a smoky, slightly sweet, and earthy vegetable complement. Flavor keywords: grilled, pepper-crusted, medium-rare, smoky, savory, juicy, charred, earthy, slightly sweet.

[검색 된 와인 리뷰]
[와인리뷰 1]
source : data/winemag-data-130k-v2.csv
row : 1728.0
content :
: 1728
country: US
description: Coming from an extreme coastal vineyard three miles north of Cambria, this cool-climate study offers aromas of blistered cherry tomatoes, Eastern European goulash spices and crushed peppercorns of all colors. It's fresh and lively on the palate, with hints of rose and tart raspberry. At the same time, it's quite vegetal, with green notes that will evolve into more balanced umami flavors from 2017 to 2022.
designation: Steiner Creek Vineyard
points: 92
price: 48
provinc

## 검색 된 리뷰를 근거로 와인 추천 생성

In [23]:
recommend_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a knowledgeable and friendly sommelier.

Your task is to recommend wines that pair well with the given dish.
Use the retrieved wine reviews as the main evidence.
Do not invent specific wines that are not supported by the retrieved reviews.
If the retrieved reviews are insufficient, say that the evidence is limited.

Respond in Korean.
"""
    ),
    (
        "human",
        """
[Dish flavor description]
{dish_flavor}

[Retrieved wine reviews]
{wine_reviews}

[Output format]
1. 추천 와인 스타일:
2. 추천 이유:
3. 근거로 사용한 리뷰 요약:
4. 주의할 점:
"""
    ),
])

recommend_llm = ChatOpenAI(
    model=TEXT_MODEL
)

recommend_wines_chain = recommend_prompt | recommend_llm | StrOutputParser()

recommendation = recommend_wines_chain.invoke({
    "dish_flavor": retrieval_result["dish_flavor"],
    "wine_reviews": retrieval_result["wine_reviews"],
})

print(recommendation)

1. 추천 와인 스타일: 페퍼리하며 중간 바디감의 시라(Syrah) 또는 풍부한 향신료와 붉은 과일이 느껴지는 피노 누아(Pinot Noir)

2. 추천 이유: 스테이크의 페퍼 크러스트와 잘 어울리는 페퍼 향과 스모키한 풍미가 있는 시라가 찰떡궁합입니다. 또한, 중간-레어 구이 스테이크의 육즙과 강렬한 맛에 풍부한 과일과 향신료가 조화를 이루는 와인이 필요합니다. 피노 누아는 상큼한 라즈베리와 약간의 허브 향이 야채의 달콤하면서도 구운 맛과도 잘 어울립니다.

3. 근거로 사용한 리뷰 요약:
- Greenwood Ridge 2013 Estate Bottled Syrah는 독특한 블랙 페퍼 캐릭터와 스모키함, 활기찬 라즈베리 노트가 스테이크의 페퍼 크러스트와 풍미를 보완해 줍니다.
- Presqu'ile 2012 Steiner Creek Vineyard Pinot Noir는 각종 후추와 블리스터 체리 토마토 향, 균형 잡힌 우마미 풍미로 구운 야채와 중간 바디 스테이크와 어울립니다.

4. 주의할 점: 리뷰에 나온 와인 대부분은 미국산이므로 다른 지역 와인을 시도할 때는 비슷한 향신료와 과일 특성을 가진 와인을 선택하는 것이 좋습니다. 또한, Chateau Ste. Michelle 2006 Syrah는 가벼운 스타일이라 강한 스테이크와 매칭 시 풍미가 약할 수 있으니 참고 바랍니다.


## 전체 RAG Chain 연결
- 이미지 URL을 음식 풍미 설명으로 변환한다.
- 음식 풍미 설명으로 Pinecone에서 관련 와인 리뷰를 검색한다.
- 검색 된 리뷰를 근거로 와인을 추천한다.

In [24]:
ai_sommelier_rag_chain = (
    describe_dish_flavor_chain
    | wine_review_retrieval_chain
    | recommend_wines_chain
)

output = ai_sommelier_rag_chain.invoke({
    "image_urls" : [
        "https://justcook.butcherbox.com/wp-content/uploads/2025/02/Rib-Eye-Steak-au-Poivre-with-Roasted-Veggies--500x500.jpg"

    ]
})

print(output)

1. 추천 와인 스타일: 블랙 페퍼 향이 풍부한 시라 품종의 중량감 있는 레드 와인

2. 추천 이유: 스테이크의 후추향과 구운, 고기풍미가 강한 맛과 잘 어울리는, 블랙 페퍼 향과 그을린 듯한 스모키한 느낌을 가진 시라 와인이 조화롭습니다. 또한 구운 채소의 약간 달콤하고 흙내음 섞인 맛에 맞춰 와인의 미묘한 산도와 스파이스가 균형을 잡아줍니다.

3. 근거로 사용한 리뷰 요약:
- Greenwood Ridge 2013 Estate Bottled Syrah (Mendocino Ridge): 블랙 페퍼 특성이 매우 뚜렷하며, 중간 바디에 다크하고 스모키한 캐릭터가 스테이크와 잘 어울립니다.
- Mesa Del Sol 2011 Syrah (Arroyo Seco): 흑백 후추, 붉은 과일 향과 함께 블랙 페퍼콘, 녹색 올리브, 검은 자두 맛이 고기의 향미와 잘 매칭됩니다.
- Four Lanterns 2013 Fire Light Syrah (Paso Robles): 향에서 후추, 라벤더, 스파이시한 느낌과 함께 훈제 고기 맛이 있어 스테이크의 강렬한 풍미와 조화롭습니다.

4. 주의할 점: 와인의 가격대가 다양하나 너무 묵직하고 타닌이 과도하게 강한 와인은 스테이크의 풍미를 압도할 수 있으니 중간 바디에 집중하는 것이 좋습니다. 또한, 스테이크에 후추가 많이 사용된 점을 고려해 블랙 페퍼 향이 포함된 와인을 선택하는 것이 포인트입니다.


## 다른 이미지로 테스트

In [25]:
test_image_urls = [
    "https://recipe1.ezmember.co.kr/cache/recipe/2020/04/20/edf37a60c3c90de68a26efce6aae53fa1.jpg",
    "https://recipe1.ezmember.co.kr/cache/recipe/2023/01/10/4a3651b8ba731f21bd63f5c5cac3ccc51.jpg",
    "https://recipe1.ezmember.co.kr/cache/recipe/2015/05/12/ea898a405bb0c70828b84b6b3ec464451.jpg"
]

test_output = ai_sommelier_rag_chain.invoke({
    "image_urls" : test_image_urls
})

print(test_output)

1. 추천 와인 스타일: 아로마틱하고 상큼한 화이트 와인 혹은 스파클링 와인

2. 추천 이유: 클래식 베이크드 치즈케이크의 크리미하고 부드러운 질감, 살짝 구운 듯한 단맛과 버터리한 그래함 크러스트, 바닐라 향미에 상큼하고 적당한 산미를 갖춘 화이트 와인이나 스파클링 와인이 조화를 이룰 것입니다. 이는 디저트의 풍부함을 균형 있게 잡아주면서 깔끔한 마무리를 선사합니다.

3. 근거로 사용한 리뷰 요약: 오레곤 윌라멧 밸리에서 생산된 Amity 2006 Riesling은 사과, 배, 라임 등 신선한 과일 향과 크리미한 질감, 미네랄리티가 어우러져 디저트와 잘 어울리는 상큼하고 텍스처감 있는 와인으로 평가받았습니다.

4. 주의할 점: 치즈케이크의 단맛이 강하기 때문에 너무 강한 바디감의 와인은 피하는 것이 좋으며, 와인 자체가 지나치게 단 와인보다는 산미가 적당히 있는 스타일이 어울립니다.  

---

1. 추천 와인 스타일: 스파이시한 맛과 감칠맛을 잘 받쳐줄 중간 체격의 레드 와인 (예: 시라)

2. 추천 이유: 떡볶이의 강한 매운맛과 달콤함, 감칠맛이 어우러진 진한 고추장 소스에는 약간 스파이시하고 약간 스모키한 풍미가 있는 시라 품종 와인이 적합합니다. 이런 와인은 매운맛과 조화를 이루고 떡볶이의 진한 소스와도 잘 어울립니다.

3. 근거로 사용한 리뷰 요약: 캘리포니아 파소 로블레스의 Four Lanterns 2013 Fire Light Syrah는 후추 향, 타르, 라벤더, 짭짤한 고기 향이 어우러지며 스파이시한 맛과 복합미로 매운 음식과 좋은 페어링을 기대할 수 있습니다.

4. 주의할 점: 너무 무겁거나 떫은 타닌이 과한 와인은 매운맛을 더욱 자극할 수 있으므로 적당한 바디감과 스파이시함을 갖춘 와인을 선택하는 것이 좋습니다.

---

1. 추천 와인 스타일: 신선한 해산물과 매운맛을 상쇄할 수 있는 상큼한 로제 와인

2. 추천 이유: 해물과 매콤한 국물의 조합이 강한 해물전골에는 상큼하고 숙성감이 너무 강하지 않은 로제 와인이 조화롭습니